# 大语言模型低比特量化

> 上一章看到，Decode 每生成一个 Token，都要读取大量模型权重。一个 7B 模型如果使用 BF16 权重，仅参数就约占 14 GB；如果机器只有 8 GB 显存，模型甚至无法完整加载。
>
> 本章只回答一个问题：**怎么让模型中的数字占更少空间，同时尽量不把效果压坏？**
>
> 我们沿着一条问题链往下走：
>
> 1. **低比特表示**：INT4 只有 16 个位置，怎么近似连续浮点数？
> 2. **权重量化**：为什么 `W4A16` 常见，GPTQ / AWQ 又在修什么误差？
> 3. **Activation 量化**：为什么 `W8A8 / FP8` 比 Weight-only 更难？
> 4. **KV Cache 量化**：长上下文下，为什么连缓存也开始压位宽？
> 5. **模型格式**：GPTQ、AWQ、GGUF、FP8 到底是在描述算法、精度还是文件格式？
>
> 读完这一章，希望你看到 Hugging Face、vLLM、llama.cpp 或厂商报告里的 `W4A16 / W8A8 / GPTQ / AWQ / SmoothQuant / FP8 / Q4_K_M` 时，能立刻知道它在系统里的位置。

先从最直接的问题开始：不同精度下，一个 7B 模型的权重究竟有多大？

In [ ]:
def model_weight_size_gb(params_billion, bits_per_weight):
    return params_billion * 1e9 * bits_per_weight / 8 / 1e9

params_billion = 7
formats = {"FP32": 32, "FP16/BF16": 16, "INT8": 8, "INT4(ideal)": 4}

for name, bits in formats.items():
    print(f"{name:<14} {model_weight_size_gb(params_billion, bits):>5.1f} GB")

INT4 的理论权重大小只有 BF16 的四分之一。

但这不等于：

```text
把每个浮点数直接 round 到 0~15
```

真正的问题是：**4 bit 只有 16 个离散位置，我们要用这 16 个位置覆盖哪一段浮点范围？**

这就引出量化最核心的两个参数：`scale` 和 `zero point`。

## 1. INT4 只有 16 个格子：Scale 到底是什么？

先考虑最简单的对称量化。

假设数据大致以 0 为中心，我们用 INT4 的 `[-7, 7]` 表示它。

```text
浮点范围:  -1.0 ----------------------- 1.0
整数位置:   -7  -6 ... -1  0  1 ...  6  7
```

相邻整数位置在浮点世界里代表多大距离，由 **scale** 决定。

\[
s = \frac{\max |x|}{7}
\]

量化：

\[
q = \operatorname{clip}(\operatorname{round}(x/s), -7, 7)
\]

反量化：

\[
\hat{x}=s q
\]

先用一组非常短的数据看看误差从哪里来。

In [ ]:
import numpy as np

def symmetric_quantize_dequantize(x, n_bits=4, axis=None):
    qmax = 2 ** (n_bits - 1) - 1
    max_abs = np.max(np.abs(x), axis=axis, keepdims=True)
    scale = np.maximum(max_abs / qmax, 1e-12)
    q = np.round(x / scale).clip(-qmax, qmax).astype(np.int32)
    x_hat = q.astype(np.float32) * scale
    return q, x_hat, scale

x = np.array([-1.0, -0.72, -0.31, 0.0, 0.18, 0.63, 1.0], dtype=np.float32)
q, x_hat, scale = symmetric_quantize_dequantize(x, n_bits=4)

print("scale =", float(scale.squeeze()))
print("原始:", x)
print("INT4:", q)
print("还原:", np.round(x_hat, 3))
print("MAE =", np.mean(np.abs(x - x_hat)))

观察量化误差时，最重要的不是记公式，而是理解这个取舍：

```text
scale 小
→ 格子更密
→ 小数值分得更细
→ 但覆盖范围更窄

scale 大
→ 覆盖范围更大
→ 但每个格子更粗
```

量化的本质就是：

> **用有限的离散位置覆盖一个连续数值分布。**

接下来马上遇到一个问题：如果数据根本不以 0 为中心呢？

## 2. Zero Point：如果数据偏向一侧怎么办？

比如 ReLU 后的 Activation 很可能大部分非负：

```text
[0.0, 0.1, 0.3, 0.8, 1.4, 2.7]
```

如果仍然用对称区间：

```text
[-7 ... 0 ... 7]
```

负数这一半位置几乎浪费了。

非对称（Affine）量化会引入 **zero point**，把整数区间平移到更合适的位置。

\[
q = \operatorname{round}(x/s)+z
\]

这里 `z` 是整数域偏移量，不是“数据最小值”。

In [ ]:
def affine_quantize_dequantize(x, n_bits=4):
    qmin, qmax = 0, 2**n_bits - 1
    x_min, x_max = float(np.min(x)), float(np.max(x))
    scale = max((x_max - x_min) / (qmax - qmin), 1e-12)
    zero_point = int(np.clip(np.round(qmin - x_min / scale), qmin, qmax))
    q = np.clip(np.round(x / scale) + zero_point, qmin, qmax).astype(np.int32)
    x_hat = scale * (q.astype(np.float32) - zero_point)
    return q, x_hat, scale, zero_point

activation = np.array([0.0, 0.1, 0.3, 0.8, 1.4, 2.7], dtype=np.float32)
q, x_hat, s, z = affine_quantize_dequantize(activation)

print(f"scale={s:.4f}, zero_point={z}")
print("原始:", activation)
print("量化:", q)
print("还原:", np.round(x_hat, 3))

现在已经知道“一个张量”怎么量化了，但真实 LLM 权重矩阵非常大。

下一问是：

> **整张矩阵真的应该共用一个 scale 吗？**

如果某一行范围很大，它会把全矩阵的 scale 拉粗，其他小权重就只能挤在少数几个整数格子里。

## 3. Per-tensor、Per-channel、Per-group：Scale 应该共享到多大范围？

常见粒度：

| 粒度 | Scale 共享范围 | 直觉 |
|---|---|---|
| Per-tensor | 整个张量 | 最简单，但容易被 Outlier 拖累 |
| Per-channel | 每个输出通道 | 更细，权重量化常见 |
| Per-group | 每组权重 | 4-bit LLM 很常见 |
| Per-token | 每个 Activation Token | 动态 Activation 量化常见 |

我们造一个“有一行特别大”的矩阵。

In [ ]:
np.random.seed(7)
W = np.random.randn(4, 16).astype(np.float32) * 0.25
W[1] *= 8.0

_, W_tensor, _ = symmetric_quantize_dequantize(W, n_bits=4)
_, W_channel, _ = symmetric_quantize_dequantize(W, n_bits=4, axis=1)

def groupwise_quantize_dequantize(W, group_size=4):
    out = np.empty_like(W)
    for r in range(W.shape[0]):
        for start in range(0, W.shape[1], group_size):
            block = W[r, start:start+group_size]
            _, restored, _ = symmetric_quantize_dequantize(block, n_bits=4)
            out[r, start:start+group_size] = restored
    return out

W_group = groupwise_quantize_dequantize(W)

for name, restored in {
    "Per-tensor": W_tensor,
    "Per-channel": W_channel,
    "Per-group": W_group,
}.items():
    print(f"{name:<12} MAE={np.mean(np.abs(W-restored)):.4f}")

这段实验解释了为什么很多 4-bit 模型名称里还会带：

```text
group_size=128
group_size=64
```

位宽只告诉你“每个值占多少 bit”。

**粒度**决定多少个值共享一个 scale。

所以看到一个“INT4 模型”，还不能直接判断它的精度和速度。你至少还要问：

- symmetric 还是 asymmetric？
- per-channel 还是 per-group？
- group size 多大？
- scale 自己用什么精度保存？
- kernel 是否原生支持？

## 4. W4A16：为什么先量化 Weight，而 Activation 仍保持 16 bit？

Transformer 最核心的线性层可以写成：

\[
Y = XW
\]

- `W`：模型权重
- `X`：输入 Activation

如果只量化权重：

```text
W4A16
│ │
│ └─ Activation 16 bit
└── Weight 4 bit
```

这叫 **Weight-only Quantization**。

为什么它常见？

因为模型权重是固定的，可以离线仔细校准；而 Activation 会随着每次输入变化，分布更难控制。

In [ ]:
configs = [
    ("BF16", 16, 16),
    ("W8A16", 8, 16),
    ("W4A16", 4, 16),
    ("W8A8", 8, 8),
]
for name, w_bits, a_bits in configs:
    w_size = model_weight_size_gb(7, w_bits)
    print(f"{name:<8} 权重理论大小≈{w_size:>4.1f} GB, Activation={a_bits} bit")

Weight-only 路线又会遇到一个问题：

> **直接 Round-to-Nearest（RTN）到 4 bit，精度可能掉得太多。**

于是 GPTQ、AWQ 这些名字出现了。

它们不是“另一种位宽”，而是在解决：

> **哪些权重的量化误差最值得被认真处理？**

## 5. RTN、GPTQ、AWQ：它们到底在优化什么？

### RTN — Round To Nearest

最朴素：

```text
算 scale
→ 除 scale
→ round
→ clip
```

快、简单，但不考虑“某个误差对模型输出到底有多重要”。

### GPTQ

GPTQ 是经典 **Post-Training Quantization (PTQ)** 方法之一。

直觉上，它不只看单个权重误差，而会利用近似二阶信息，在逐列/逐块量化时补偿误差。

可以把它理解成：

> “这个权重被量坏了，后面的权重能不能稍微调整，把输出误差补回来？”

### AWQ — Activation-aware Weight Quantization

AWQ 的核心直觉是：

> 有些 Weight Channel 对真实 Activation 更重要，优先保护这些通道。

它虽然量的是 **Weight**，但会利用 Activation 统计来判断哪些权重值得保护。

所以：

```text
GPTQ / AWQ
≠ 位宽名称
≠ 文件格式
= 量化方法 / 校准方法
```

## 6. Outlier：为什么 8-bit Activation 也会很难？

Weight 通常相对稳定，但 Activation 是输入相关的。

更麻烦的是，LLM Activation 中可能出现少数非常大的 **Outlier**。

例如：

```text
[0.2, -0.1, 0.4, 0.3, 18.0]
```

如果整组数据共用一个 scale，`18.0` 会把范围拉得很宽，前面几个小值就挤在非常粗的量化格子里。

In [ ]:
normal = np.array([0.2, -0.1, 0.4, 0.3, 0.5], dtype=np.float32)
with_outlier = np.array([0.2, -0.1, 0.4, 0.3, 18.0], dtype=np.float32)

for name, arr in [("normal", normal), ("with outlier", with_outlier)]:
    _, restored, scale = symmetric_quantize_dequantize(arr, n_bits=8)
    print(f"{name:<12} scale={float(scale.squeeze()):.4f}  MAE={np.mean(np.abs(arr-restored)):.4f}")

这就是为什么 `W8A8` 不是简单一句“权重 8bit、Activation 8bit”就结束了。

它真正的难点是：

> **Activation Distribution 会随输入变化，而且 Outlier 会污染 scale。**

SmoothQuant 就是为这个问题服务的经典思路之一。

## 7. SmoothQuant：把 Activation 的难题搬一点给 Weight

SmoothQuant 的名字很直观：让 Activation “更平滑”。

它利用一个等价变换，把一部分 Activation 的尺度压力迁移到 Weight：

```text
原来:
Activation 有大 Outlier
Weight 相对稳定

重新缩放后:
Activation 更容易量化
Weight 承担一部分尺度变化
```

这样更容易做 `W8A8`。

所以看到厂商写：

```text
INT8 SmoothQuant
```

应该想到：

> 这不是“另一个 INT8 格式”，而是在处理 Activation Outlier。

## 8. FP8：为什么不是所有低比特都用整数？

INT8 / INT4 用整数网格表示数值。

FP8 则仍然是浮点格式，只是位数更少。

常见 FP8 格式会在：

```text
指数范围
vs
尾数精度
```

之间做不同取舍。

它的优势在于现代 GPU 对 FP8 GEMM 有专门硬件支持，因此在训练和推理中越来越常见。

所以看到：

```text
FP8 Weight
FP8 Activation
FP8 KV Cache
```

要继续问：

- 哪一种 FP8 格式？
- static 还是 dynamic scale？
- per-tensor / per-channel / block-wise？
- GPU 和 kernel 是否真正支持？

## 9. KV Cache 也能量化吗？

上一章已经看到 KV Cache 会随着：

```text
batch × context length × layers × KV heads
```

不断增长。

长上下文和高并发服务里，KV Cache 甚至可能成为主要显存开销。

因此可以进一步做：

```text
BF16 KV
→ FP8 KV
→ INT8 / 更低位 KV
```

但 KV Cache 和 Weight 不一样：

- Weight 是静态的，可离线处理。
- KV 是运行时生成的，每个请求都不同。
- KV 精度会直接影响后续所有 Token 的 Attention。

所以 KV Cache 量化通常要在**显存容量、带宽、质量**之间做更谨慎的权衡。

In [ ]:
def kv_cache_size_gb(layers, kv_heads, head_dim, seq_len, batch, bits):
    values = 2 * layers * kv_heads * head_dim * seq_len * batch  # K + V
    return values * bits / 8 / 1e9

for bits in [16, 8, 4]:
    size = kv_cache_size_gb(
        layers=32, kv_heads=8, head_dim=128,
        seq_len=32768, batch=8, bits=bits
    )
    print(f"{bits:2d}-bit KV Cache ≈ {size:.2f} GB")

## 10. PTQ 和 QAT：量化是在训练前还是训练后做？

### PTQ — Post-Training Quantization

模型已经训练完，再做量化。

特点：

```text
成本低
不需要完整重新训练
GPTQ / AWQ / SmoothQuant 常属于这个大类
```

### QAT — Quantization-Aware Training

训练过程中就模拟量化误差，让模型主动适应低精度。

特点：

```text
成本更高
但可以进一步恢复精度
```

看到招聘 JD 写：

```text
PTQ / QAT experience
```

它问的不是某个具体库，而是：

> 你是否理解“训练后压缩”和“训练中适配低精度”这两条路线。

## 11. GPTQ、AWQ、GGUF、Q4_K_M：名字为什么容易混？

这些名字不在同一层。

### GPTQ / AWQ

主要是 **量化算法 / 校准方法**。

### W4A16 / W8A8

主要是 **Weight / Activation 位宽配置**。

### GGUF

是 llama.cpp 生态常见的 **模型文件格式 / 容器格式**。

GGUF 文件名里常出现：

```text
Q4_K_M
Q5_K_M
Q8_0
```

这些名字描述 llama.cpp 生态中的具体量化方案和 block layout。

所以不要把：

```text
AWQ vs GGUF
```

当成完全同类的二选一。

一个更好的问题是：

> 我用什么量化方法得到权重？  
> 最终要在哪个推理引擎运行？  
> 那个引擎支持什么模型格式和 kernel？

## 12. 真正部署时怎么选？

可以先按运行环境粗分。

### GPU Server：vLLM / SGLang

常见选择：

```text
BF16 / FP16
FP8
AWQ
GPTQ
部分 INT8 / Weight-only formats
```

重点看 GPU 架构和 kernel 支持。

### Local / CPU / Mac：llama.cpp

常见：

```text
GGUF
Q4_K_M
Q5_K_M
Q8_0
```

重点是内存占用、CPU/GPU offload 和本地吞吐。

所以：

> “哪个量化最好？”没有脱离硬件和推理引擎的统一答案。

## 13. 把量化放回推理链

```text
Model Weights
   ↓ Quantization
W4 / W8 / FP8
   ↓
Decode 每一步少搬数据
   ↓
更低显存占用 / 更高带宽利用

Activation
   ↓
W8A8 / FP8 / SmoothQuant

KV Cache
   ↓
FP8 / INT8 KV
   ↓
更高并发 / 更长 Context
```

下一章继续追问另一个完全不同的瓶颈：

> 即使每一步已经变便宜，自回归生成仍然一次只能确认一个 Token。  
> 能不能让一次 Target Forward 确认多个 Token？

这就是 **Speculative Decoding**。

## 小结

看到量化相关名词时，按四层拆：

1. **量什么**：Weight / Activation / KV Cache。
2. **多少 bit**：W4A16 / W8A8 / FP8。
3. **什么粒度**：per-tensor / channel / group / token。
4. **怎么控制误差**：RTN / GPTQ / AWQ / SmoothQuant / QAT。

另外：

- **GPTQ / AWQ**：算法路线。
- **GGUF**：格式 / 生态。
- **Q4_K_M**：llama.cpp 生态中的具体量化类型。
- **FP8**：低精度浮点，不等于 INT8。

## 作业

1. 计算 70B 模型 BF16 / INT8 / INT4 的理论权重大小。
2. 解释为什么 `W4A16` 比 `W4A4` 更容易落地。
3. 用一句话分别解释 GPTQ、AWQ、SmoothQuant 在“修什么问题”。
4. 为什么 KV Cache 量化和 Weight 量化的难点不同？
5. 在 Hugging Face 看到 `AWQ W4A16` 时，分别指出 AWQ 和 W4A16 描述的是哪一层信息。